In [1]:
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import BigBirdTokenizer, BigBirdModel
from transformers import get_linear_schedule_with_warmup
import numpy as np
import pandas as pd
import os, json

torch.backends.cudnn.benchmark = True

/data/shubham/miniconda/envs/_env_IBPS/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/shubham/miniconda/envs/_env_IBPS/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
with open('Data/final_train_df.pkl', 'rb') as f:
    train_df = pickle.load(f)
    train_df.rename(columns={'xgb_oof_pred': 'xgb_pred'}, inplace=True)

with open('Data/final_val_df.pkl', 'rb') as f:
    val_df = pickle.load(f)

with open('Data/final_test_df.pkl', 'rb') as f:
    test_df = pickle.load(f)

In [3]:
# Find the median of days_in_custody from train_df
median_days = train_df['days_in_custody'].median()

# Fill NaN values in all three dataframes with the median
train_df.fillna({'days_in_custody': median_days}, inplace=True)
val_df.fillna({'days_in_custody': median_days}, inplace=True)
test_df.fillna({'days_in_custody': median_days}, inplace=True)

In [4]:
# Find medians for age columns in train_df
median_min_age = train_df['min_age'].median()
median_max_age = train_df['max_age'].median()
median_median_age = train_df['median_age'].median()

# Fill NaN values in all three dataframes
for df in [train_df, val_df, test_df]:
    df.fillna({'min_age': median_min_age}, inplace=True)
    df.fillna({'max_age': median_max_age}, inplace=True)
    df.fillna({'median_age': median_median_age}, inplace=True)

In [5]:
train_df.columns

Index(['CNR', 'bail_type', 'details', 'days_in_custody_available',
       'days_in_custody', 'age_available', 'min_age', 'max_age', 'median_age',
       'shap_sum_pos', 'shap_sum_neg', 'shap_max_pos', 'shap_min_neg',
       'shap_pos_count', 'shap_neg_count', 'shap_l1_total',
       'shap_top3_abs_sum', 'xgb_pred', 'outcome'],
      dtype='object')

In [6]:
scaler = StandardScaler()

columns_to_standardize = ['days_in_custody', 'min_age', 'max_age', 'median_age',
                          'shap_sum_pos', 'shap_sum_neg', 'shap_max_pos', 'shap_min_neg',
                          'shap_pos_count', 'shap_neg_count', 'shap_l1_total',
                          'shap_top3_abs_sum']
                          
train_df[columns_to_standardize] = scaler.fit_transform(train_df[columns_to_standardize])
val_df[columns_to_standardize] = scaler.transform(val_df[columns_to_standardize])
test_df[columns_to_standardize] = scaler.transform(test_df[columns_to_standardize])

In [7]:
SCALAR_COLS = [
    "bail_type",
    "days_in_custody_available",
    "days_in_custody",
    "age_available",
    "min_age",
    "max_age",
    "median_age",
    # "shap_sum_pos",
    # "shap_sum_neg",
    # "shap_max_pos",
    # "shap_min_neg",
    # "shap_pos_count",
    # "shap_neg_count",
    # "shap_l1_total",
    # "shap_top3_abs_sum",
    # "xgb_pred",
]

In [8]:
class ScalarDataset(Dataset):
    def __init__(self, df):
        self.X = df[SCALAR_COLS].astype(np.float32).to_numpy()
        self.y = df['outcome'].astype(np.float32).to_numpy()

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return {
            'x': torch.from_numpy(self.X[idx]),
            'y': torch.tensor(self.y[idx])
        }

In [9]:
class TinyMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


In [10]:
train_scalar_ds = ScalarDataset(train_df)
val_scalar_ds = ScalarDataset(val_df)

train_scalar_loader = DataLoader(train_scalar_ds, batch_size=128, shuffle=True)
val_scalar_loader = DataLoader(val_scalar_ds, batch_size=128, shuffle=False)

In [11]:
def train_one_epoch_scalar(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_scalar(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_logits = []
    all_labels = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X_train = train_df[SCALAR_COLS].values
y_train = train_df["outcome"].values

X_val = val_df[SCALAR_COLS].values
y_val = val_df["outcome"].values

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_train, y_train)

val_probs = clf.predict_proba(X_val)[:,1]
val_preds = (val_probs >= 0.5).astype(int)

print("LogReg acc:", accuracy_score(y_val, val_preds))
print("LogReg f1 :", f1_score(y_val, val_preds))


LogReg acc: 0.5775156475686086
LogReg f1 : 0.582639714625446


In [13]:
class LinearBaseline(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, 1)

    def forward(self, x):
        return self.fc(x).squeeze(1)


In [14]:
# =========================
# BASELINE 2 — Linear Torch
# =========================

print("\n==============================")
print("Baseline 2: Linear Torch Model")
print("==============================")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_linear = LinearBaseline(len(SCALAR_COLS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model_linear.parameters(),
    lr=1e-3,
    weight_decay=1e-2   # 🔴 strong regularization
)

best_f1 = 0
best_state = None

for epoch in range(50):
    train_loss = train_one_epoch_scalar(model_linear, train_scalar_loader, optimizer, criterion, device)
    metrics = eval_scalar(model_linear, val_scalar_loader, criterion, device)

    print(f"[Linear] Epoch {epoch:02d} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={metrics['loss']:.4f} | "
          f"acc={metrics['accuracy']:.4f} | "
          f"f1={metrics['f1']:.4f}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_state = model_linear.state_dict()

# Load best
model_linear.load_state_dict(best_state)

print("\n✅ Best Linear Baseline F1:", best_f1)



Baseline 2: Linear Torch Model
[Linear] Epoch 00 | train_loss=0.6997 | val_loss=0.6851 | acc=0.5703 | f1=0.5256
[Linear] Epoch 01 | train_loss=0.6836 | val_loss=0.6814 | acc=0.5816 | f1=0.5810
[Linear] Epoch 02 | train_loss=0.6813 | val_loss=0.6793 | acc=0.5816 | f1=0.5852
[Linear] Epoch 03 | train_loss=0.6800 | val_loss=0.6784 | acc=0.5838 | f1=0.5869
[Linear] Epoch 04 | train_loss=0.6791 | val_loss=0.6777 | acc=0.5831 | f1=0.5849
[Linear] Epoch 05 | train_loss=0.6785 | val_loss=0.6775 | acc=0.5833 | f1=0.5822
[Linear] Epoch 06 | train_loss=0.6780 | val_loss=0.6771 | acc=0.5831 | f1=0.5816
[Linear] Epoch 07 | train_loss=0.6776 | val_loss=0.6767 | acc=0.5847 | f1=0.5902
[Linear] Epoch 08 | train_loss=0.6771 | val_loss=0.6758 | acc=0.5859 | f1=0.5883
[Linear] Epoch 09 | train_loss=0.6768 | val_loss=0.6761 | acc=0.5845 | f1=0.5902
[Linear] Epoch 10 | train_loss=0.6766 | val_loss=0.6753 | acc=0.5850 | f1=0.5928
[Linear] Epoch 11 | train_loss=0.6763 | val_loss=0.6756 | acc=0.5835 | f1=0.5

In [15]:
# =========================
# BASELINE 3 — Tiny MLP
# =========================

print("\n==============================")
print("Baseline 3: Tiny MLP")
print("==============================")

model_mlp = TinyMLP(len(SCALAR_COLS)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model_mlp.parameters(),
    lr=3e-4,
    weight_decay=5e-2   # 🔴 VERY strong regularization
)

best_f1 = 0
best_state = None
bad_epochs = 0
patience = 8

for epoch in range(50):
    train_loss = train_one_epoch_scalar(model_mlp, train_scalar_loader, optimizer, criterion, device)
    metrics = eval_scalar(model_mlp, val_scalar_loader, criterion, device)

    print(f"[TinyMLP] Epoch {epoch:02d} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={metrics['loss']:.4f} | "
          f"acc={metrics['accuracy']:.4f} | "
          f"f1={metrics['f1']:.4f}")

    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_state = model_mlp.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("🛑 Early stopping TinyMLP")
        break

# Load best
model_mlp.load_state_dict(best_state)

print("\n✅ Best TinyMLP Baseline F1:", best_f1)



Baseline 3: Tiny MLP
[TinyMLP] Epoch 00 | train_loss=0.7008 | val_loss=0.6907 | acc=0.5207 | f1=0.6402
[TinyMLP] Epoch 01 | train_loss=0.6933 | val_loss=0.6884 | acc=0.5159 | f1=0.5739
[TinyMLP] Epoch 02 | train_loss=0.6904 | val_loss=0.6863 | acc=0.5190 | f1=0.5806
[TinyMLP] Epoch 03 | train_loss=0.6885 | val_loss=0.6848 | acc=0.5166 | f1=0.5855
[TinyMLP] Epoch 04 | train_loss=0.6870 | val_loss=0.6837 | acc=0.5833 | f1=0.5858
[TinyMLP] Epoch 05 | train_loss=0.6854 | val_loss=0.6822 | acc=0.5847 | f1=0.5904
[TinyMLP] Epoch 06 | train_loss=0.6850 | val_loss=0.6815 | acc=0.5818 | f1=0.5937
[TinyMLP] Epoch 07 | train_loss=0.6841 | val_loss=0.6805 | acc=0.5840 | f1=0.5894
[TinyMLP] Epoch 08 | train_loss=0.6825 | val_loss=0.6793 | acc=0.5850 | f1=0.5858
🛑 Early stopping TinyMLP

✅ Best TinyMLP Baseline F1: 0.6401590457256461
